# Deep learning III: deep neural networks

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

Two shallow networks are composed, written as one deep network with two hidden layers (in NumPy with matrices, then in PyTorch), and compared with shallow networks by their linear regions per parameter. At the end a deep network is trained. There is nothing to type - run the cells in order and compare the output with the lecture.

Later cells use the variables and functions of earlier ones: Runtime > Run all runs everything at once. The notebook needs nothing from the earlier notebooks.

## Code-along 1. Two networks in code (demo)

Two shallow networks with three hidden units each. `shallow_1_1_3` takes the input, the four phi and the six theta, and returns the output. Each tuple holds the ten parameters of one network; `*net1` unpacks it into the arguments. Network 1 gets inputs between 0 and 1, network 2 inputs between -1 and 1: the range of the outputs of network 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ReLU(preactivation):
    """Rectified linear unit: negative values become zero"""
    return np.maximum(0, preactivation)

def shallow_1_1_3(x, phi_0, phi_1, phi_2, phi_3,
                  theta_10, theta_11, theta_20, theta_21, theta_30, theta_31):
    """Shallow network: 1 input, 1 output, 3 hidden units"""
    # activations: three lines, each passed through the ReLU
    h_1 = ReLU(theta_10 + theta_11 * x)
    h_2 = ReLU(theta_20 + theta_21 * x)
    h_3 = ReLU(theta_30 + theta_31 * x)
    # weighted sum of the activations plus the offset phi_0
    return phi_0 + phi_1 * h_1 + phi_2 * h_2 + phi_3 * h_3

In [ ]:
# order: phi_0, phi_1, phi_2, phi_3, then theta_10, theta_11,
# theta_20, theta_21, theta_30, theta_31
net1 = (-1.0, 6.0, -4.0, 4.0,   0.0, 1.0,  -1.0, 3.0,  -2.0, 3.0)
net2 = (-1.0, 2.0, -1.5, 2.0,   1.0, 1.0,   1.0, 2.0,  -1.0, 2.0)

# inputs of network 1: 1201 values between 0 and 1
x = np.linspace(0, 1, 1201)
# output of network 1
y = shallow_1_1_3(x, *net1)
# smallest and largest output
print('Network 1: y from', y.min(), 'to', y.max())

# inputs of network 2: 1201 values between -1 and 1
y_in = np.linspace(-1, 1, 1201)
# output of network 2
y_out = shallow_1_1_3(y_in, *net2)
print('Network 2: y\' from', y_out.min(), 'to', y_out.max())

In [ ]:
def plot_function(ax, x, y, xlabel, ylabel, title):
    """Draw the curve y over x into the panel ax"""
    ax.plot(x, y)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    # same vertical range in all plots
    ax.set_ylim([-1.1, 1.1])
    ax.set_title(title)

In [ ]:
# one figure with two panels
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
plot_function(ax[0], x, y, 'Input, $x$', 'Output, $y$', 'Network 1')
plot_function(ax[1], y_in, y_out, 'Input, $y$', "Output, $y'$", 'Network 2')
fig.tight_layout()
plt.show()

## Code-along 2. Compose them (demo)

The output of network 1 is the input of network 2. `count_regions` counts the changes of slope along the curve and adds one. How many linear regions do you expect for the composed function? Decide before you run the cell.

In [ ]:
# the output of network 1 is the input of network 2
y = shallow_1_1_3(x, *net1)
y_prime = shallow_1_1_3(y, *net2)

def count_regions(x, y):
    """Number of linear regions of the curve y over x"""
    # slope between each pair of neighboring points
    slopes = np.round(np.diff(y) / np.diff(x), 6)
    # one more region than changes of slope
    return np.sum(np.diff(slopes) != 0) + 1

print('Network 1:', count_regions(x, y), 'linear regions')
print('Network 2:', count_regions(y_in, y_out), 'linear regions')
print('Composed: ', count_regions(x, y_prime), 'linear regions')

In [ ]:
# the composed function: from x to y'
fig, ax = plt.subplots(figsize=(6, 3.6))
plot_function(ax, x, y_prime, 'Input, $x$', "Output, $y'$",
              'Network 1, then network 2')
fig.tight_layout()
plt.show()

## Code-along 3. Change the first network (demo)

Network 1 as a rising line (y = -1 + 2x), and as a constant (y = 0.5). What is the composed function in each case? Decide before you run the cell.

In [ ]:
# network 1 as a rising line: y = -1 + 2x
line = (-1.0, 2.0, 0.0, 0.0,   0.0, 1.0,  -1.0, 3.0,  -2.0, 3.0)
# network 1 as a constant: y = 0.5
flat = (0.5, 0.0, 0.0, 0.0,   0.0, 1.0,  -1.0, 3.0,  -2.0, 3.0)

# compose each one with network 2
y_line = shallow_1_1_3(shallow_1_1_3(x, *line), *net2)
y_flat = shallow_1_1_3(shallow_1_1_3(x, *flat), *net2)
print('Line, then network 2:    ', count_regions(x, y_line), 'linear regions')
print('Constant, then network 2:', count_regions(x, y_flat), 'linear region')

In [ ]:
# the two composed functions side by side
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
plot_function(ax[0], x, y_line, 'Input, $x$', "Output, $y'$",
              'Network 1 is a line')
plot_function(ax[1], x, y_flat, 'Input, $x$', "Output, $y'$",
              'Network 1 is a constant')
fig.tight_layout()
plt.show()

## Code-along 4. The deep network with matrices (demo)

The two composed networks as one deep network with two hidden layers. The offsets `beta_1` and the slopes `Omega_1` of the second layer are computed from the parameters of the two networks; `@` is the matrix product. The network itself is three lines: two hidden layers and the output. The last line checks that it computes the same function as the two composed networks.

In [ ]:
# network 2, input side: offsets and slopes as columns
theta_p0 = np.array([[1.0], [1.0], [-1.0]])
theta_p1 = np.array([[1.0], [2.0], [2.0]])
# network 1, output side: offset and weights (as a row)
phi_0 = -1.0
phi = np.array([[6.0, -4.0, 4.0]])

# second layer: offsets (3 x 1) and slopes (3 x 3)
beta_1 = theta_p0 + theta_p1 * phi_0
Omega_1 = theta_p1 @ phi
print(beta_1.flatten())
print(Omega_1)

In [ ]:
# first layer: offsets and slopes of network 1
beta_0 = np.array([[0.0], [-1.0], [-2.0]])
Omega_0 = np.array([[1.0], [3.0], [3.0]])
# output: offset and weights of network 2
beta_2 = np.array([[-1.0]])
Omega_2 = np.array([[2.0, -1.5, 2.0]])

def deep_network(x, beta_0, Omega_0, beta_1, Omega_1, beta_2, Omega_2):
    """Deep network with two hidden layers, in matrix notation"""
    # one column per input
    X = x.reshape(1, -1)
    # first hidden layer
    h_1 = ReLU(beta_0 + Omega_0 @ X)
    # second hidden layer
    h_2 = ReLU(beta_1 + Omega_1 @ h_1)
    # output
    y = beta_2 + Omega_2 @ h_2
    print('Shapes:', X.shape, h_1.shape, h_2.shape, y.shape)
    return y.flatten()

y_deep = deep_network(x, beta_0, Omega_0, beta_1, Omega_1, beta_2, Omega_2)
# same function as the two composed networks?
print('Same as composed:', np.allclose(y_deep, y_prime))

## Code-along 5. The deep network in PyTorch (demo)

The same network in PyTorch: one more `nn.Linear` and `nn.ReLU` in the chain. Count the parameters before you run the first cell. The second cell loads our matrices and compares the output with that of the two composed networks. PyTorch wants one row per input: `reshape(-1, 1)`.

In [ ]:
# PyTorch and its module with the building blocks of networks
import torch
from torch import nn

# two hidden layers of three hidden units each
model = nn.Sequential(
    nn.Linear(1, 3),
    nn.ReLU(),
    nn.Linear(3, 3),
    nn.ReLU(),
    nn.Linear(3, 1),
)
# every group of parameters: name, shape, number of values
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.numel())
# all parameters of the network
print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
def to_tensor(a):
    """NumPy array as a PyTorch tensor of 32-bit floats"""
    return torch.tensor(a, dtype=torch.float32)

# overwrite the random starting parameters with our matrices
with torch.no_grad():
    model[0].weight[:] = to_tensor(Omega_0)
    model[0].bias[:] = to_tensor(beta_0.flatten())
    model[2].weight[:] = to_tensor(Omega_1)
    model[2].bias[:] = to_tensor(beta_1.flatten())
    model[4].weight[:] = to_tensor(Omega_2)
    model[4].bias[:] = to_tensor(beta_2.flatten())

# the 1201 inputs as a column: one row per input
X_grid = to_tensor(x).reshape(-1, 1)
# inference only
with torch.no_grad():
    y_torch = model(X_grid).numpy().flatten()
print('Same as composed:', np.allclose(y_torch, y_prime, atol=1e-5))

## Code-along 6. Shallow vs. deep: regions per parameter (demo)

One input, one output, K hidden layers of D hidden units each: the number of parameters and the largest possible number of linear regions. The last two rows spend about 470 parameters in two ways: one layer of 157 hidden units, or five layers of 10.

In [ ]:
def parameters(D, K):
    """Parameters: 1 input, 1 output, K hidden layers of D units"""
    return 3 * D + 1 + (K - 1) * D * (D + 1)

def max_regions(D, K):
    """Largest possible number of linear regions"""
    return (D + 1) ** K

# (D, K): hidden units per layer, number of hidden layers
for D, K in [(3, 1), (3, 2), (157, 1), (10, 5)]:
    print(f'D = {D:3d}, K = {K}: {parameters(D, K):3d} parameters, '
          f'up to {max_regions(D, K):,} linear regions')

## Code-along 7. Train a deep network (demo)

The 20 training points and the training function of the last lecture, then a deep network with two hidden layers of ten hidden units, trained from a random start. Compare the final loss with the 0.082 of the shallow network with 50 hidden units (151 parameters).

In [ ]:
# the twenty training points of the last lecture
x_train = np.array([0.09291784,0.46809093,0.93089486,0.67612654,0.73441752,
                    0.86847339,0.49873225,0.51083168,0.18343972,0.99380898,
                    0.27840809,0.38028817,0.12055708,0.56715537,0.92005746,
                    0.77072270,0.85278176,0.05315950,0.87168699,0.58858043])
y_train = np.array([-0.15934537,0.18195445,0.45127015,0.13921448,0.09366691,
                    0.30567674,0.37229117,0.40716968,-0.08131792,0.41187806,
                    0.36943738,0.3994327,0.01906257,0.35820410,0.45256496,
                    -0.0183121,0.02957665,-0.24354444,0.14803884,0.26824970])
# as tensors: one row per training point, one column
X = to_tensor(x_train).reshape(-1, 1)
Y = to_tensor(y_train).reshape(-1, 1)

# least-squares loss: sum of the squared differences
loss_fn = nn.MSELoss(reduction='sum')

def train(model, X, Y, steps=2000, lr=0.01):
    """Train the model on (X, Y); return the final loss"""
    # the optimizer changes the parameters; lr is the step size
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps + 1):
        # run the network and compute the loss
        loss = loss_fn(model(X), Y)
        if step % 1000 == 0:
            print(f'Step {step:5d}: loss = {loss.item():.3f}')
        # clear the old directions, compute the new ones
        optimizer.zero_grad()
        loss.backward()
        # change every parameter by a small step
        optimizer.step()
    return loss_fn(model(X), Y).item()

In [ ]:
# same random starting parameters for everyone
torch.manual_seed(0)
# hidden units per layer
D = 10
deep = nn.Sequential(nn.Linear(1, D), nn.ReLU(),
                     nn.Linear(D, D), nn.ReLU(),
                     nn.Linear(D, 1))
print('Parameters:', sum(p.numel() for p in deep.parameters()))
# 5000 training steps from the random start
loss_deep = train(deep, X, Y, steps=5000)

In [ ]:
# 100 inputs between 0 and 1, as a column
x_plot = torch.linspace(0, 1, 100).reshape(-1, 1)
# inference only
with torch.no_grad():
    y_plot = deep(x_plot)
# the trained network with the training points (magenta)
fig, ax = plt.subplots(figsize=(6, 3.6))
plot_function(ax, x_plot.numpy(), y_plot.numpy(), 'Input, $x$',
              'Output, $y$', f'K = 2, D = 10: loss = {loss_deep:.3f}')
ax.plot(x_train, y_train, 'mo')
fig.tight_layout()
plt.show()